In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
})

## Load and Visualize the data

In [ ]:
train_path = '../data/processed/arousals/arousals_train.npz'
test_path = '../data/processed/arousals/arousals_test.npz'

def load_data(npz_path):
    data = np.load(npz_path)
    signals = data['eeg_windows']
    contexts = data['context_windows']
    labels = data['labels']

    return signals, contexts, labels

signals, contexts, labels = load_data(train_path)

In [ ]:
arousals = contexts[labels == 1]
non_arousals = contexts[labels == 0]

print("Signals Events: ", signals.shape)
print("Context shape: ", contexts.shape)
print("Arousal Events: ", arousals.shape)
print("Non Arousal Events: ", non_arousals.shape)

## View STFT Features 

In [ ]:
import seaborn as sns

# 2. Select a sample to inspect (Change this index to view different windows)
sample_idx = 0  
label_dict = {0: "No Arousal (Pure Sleep)", 1: "Arousal Onset Zone"}

# Isolate the sample components
raw_signal = signals[sample_idx]  # Shape: (2, 1500)
context_matrix = contexts[sample_idx]  # Shape: (148, 10)
sample_label = labels[sample_idx]

print(f"[+] Visualizing Sample #{sample_idx} | True Label: {label_dict[sample_label]}")

# 3. Create the multi-plot canvas
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 2.2]})

# -----------------------------------------------------------------------------
# UPPER PLOT: Raw EEG Signal Branch (15 Seconds)
# -----------------------------------------------------------------------------
fs = 100  # Assumed 100Hz sampling rate
time_axis_local = np.linspace(0, 15, raw_signal.shape[1])

axes[0].plot(time_axis_local, raw_signal[0], label="C3-M2 (Raw)", color="tab:blue", alpha=0.8, linewidth=1)
axes[0].plot(time_axis_local, raw_signal[1], label="C4-M1 (Raw)", color="tab:purple", alpha=0.6, linewidth=1)
axes[0].set_title(f"Branch 1 Input: Local Raw Signal Window ({label_dict[sample_label]})", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Normalized Amp")
axes[0].set_xlabel("Local Time (Seconds)")
axes[0].set_xlim(0, 15)
axes[0].grid(True, linestyle=":", alpha=0.5)
axes[0].legend(loc="upper right")

# -----------------------------------------------------------------------------
# LOWER PLOT: Rolling Spectral Context Branch (5 Minutes History)
# -----------------------------------------------------------------------------
# Transpose back to (Channels/Bands, TimeSteps) so Time moves left-to-right on the X-axis
context_t = context_matrix.T  # Shape: (10, 148)

# Define clean feature labels for your 10 channels (5 bands * 2 channels)
yticklabels = [
    "C3 Delta", "C3 Theta", "C3 Alpha", "C3 Sigma", "C3 Beta",
    "C4 Delta", "C4 Theta", "C4 Alpha", "C4 Sigma", "C4 Beta"
]

# Create an informative time axis mapping the 148 rolling steps across 300 seconds
time_axis_context = np.linspace(-300, 0, context_matrix.shape[0])

# Plot using a clean heatmap layout
sns.heatmap(
    context_t, 
    ax=axes[1], 
    cmap="viridis", 
    cbar_kws={'label': 'Power Spectrum Density (PSD) Magnitude'},
    yticklabels=yticklabels
)

# Clean up X-axis tick marks to show clean negative lookback times
num_ticks = 10
tick_indices = np.linspace(0, context_matrix.shape[0] - 1, num_ticks, dtype=int)
tick_labels = [f"{int(time_axis_context[i])}s" for i in tick_indices]
axes[1].set_xticks(tick_indices)
axes[1].set_xticklabels(tick_labels)

axes[1].set_title("Branch 2 Input: 5-Minute Rolling Sleep Macro-Architecture Context Timeline", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Historical Time Buffer Relative to Local Window Onset (0s = Start of Local Window)")
axes[1].set_ylabel("EEG Channel Frequency Bands")

plt.tight_layout()
plt.show()

## TSNE representation

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# Assuming your context arrays are separated like your signals:
# context_arousals shape: (15677, 149, 10)
# context_non shape: (15364, 149, 10)

n_samples_per_class = 2000

idx_arousal = np.random.choice(len(arousals), n_samples_per_class, replace=False)
idx_non = np.random.choice(len(non_arousals), n_samples_per_class, replace=False)

sub_arousal = arousals[idx_arousal]
sub_non = non_arousals[idx_non]

# Combine: (4000, 149, 10)
X_combined = np.vstack([sub_arousal, sub_non]) 
y = np.hstack([np.ones(n_samples_per_class), np.zeros(n_samples_per_class)])

print("[*] Flattening context features...")
# Flatten (149 time steps * 10 bands) into 1490 flat features
X_flat = X_combined.reshape(X_combined.shape[0], -1) 

print("[*] Scaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

print("[*] Computing t-SNE embedding...")
tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_jobs=-1, init='pca')
X_tsne = tsne.fit_transform(X_scaled)

# Separate the projections
tsne_arousal = X_tsne[y == 1]
tsne_non = X_tsne[y == 0]

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(tsne_non[:, 0], tsne_non[:, 1], color='tab:blue', alpha=0.4, s=10, label='Non-Arousal')
ax.scatter(tsne_arousal[:, 0], tsne_arousal[:, 1], color='tab:orange', alpha=0.6, s=12, label='Arousal')

ax.set_title('t-SNE Manifold: Context Bandpower Features', weight='semibold', pad=12)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(loc='upper right', markerscale=2)

plt.tight_layout()
plt.savefig('context_tsne.png', dpi=300)
plt.show()